# EggProducingChickens_Exploration.ipynb
**ICS 3202 — Artificial Intelligence | Project Deliverable 1**

**Project:** An Integrated Enterprise Resource Planning System for Poultry Farm Management

**Dataset:** [Egg Producing Chickens](https://www.kaggle.com/datasets/phuzoman/egg-producing-chickens) (Kaggle, `phuzoman/egg-producing-chickens`)
1,000 daily observations across chicken breeds, recording physical attributes, feed intake, sunlight exposure, and eggs laid per day. Note: partially artificially generated from domain knowledge rather than raw farm sensor data — suitable for this exploration exercise, worth supplementing with real farm data later in the project.

**Expected output variable of the final application:** `EggsPerDay` — the number of eggs a bird is expected to lay on a given day, predicted from feed intake, age, weight, breed, and sunlight exposure. This feeds the ERP's **Production Tracking** module (forecast expected yield vs. actual, flag underperforming flocks) and **Feed Management** module (relate feed spend to output).

## Dataset Discovery (Instructions 1 & 2)

Open-source datasets considered as relevant to the poultry ERP's ML engine, and the output variable each would support:

| Dataset | Source | Candidate output variable |
|---|---|---|
| **Egg Producing Chickens** (selected) | Kaggle: `phuzoman/egg-producing-chickens` | `EggsPerDay` — daily egg yield per bird |
| Prediction of Egg Production Rate in Poultry | Kaggle: `deepikabidri/prediction-of-egg-production-rate-in-poultry` | egg production rate, from humidex/air/water quality |
| Environmental Effect on Egg Production | Kaggle: `faysal1998/environmental-effect-on-egg-production` | egg production rate, from environmental conditions |
| Poultry Farm Management Dataset (Sabarhi Hatcheries) | IEEE DataPort | weekly mortality rate / cull rate, from feed & medicine records |
| Eggs and Butter (USDA-style historical) | Kaggle: `datasciencedonut/eggs-and-butter` | national egg production trend (too aggregate for a single-farm ERP) |

**Why `Egg Producing Chickens` was selected:** it is the only candidate that is (a) a clean single CSV rather than aggregate/national statistics, (b) has an output variable (`EggsPerDay`) that maps directly onto the ERP's own **Production Tracking** and **Feed Management** modules, and (c) includes enough predictor variables (feed intake, age, weight, breed, sunlight exposure) to support a genuine regression model rather than a toy example. Its main limitation, addressed below, is that it is partly synthetically generated.

## Setup — download the dataset

In [ ]:
# Kaggle will not serve a dataset without an API token, so kaggle.json is
# uploaded first. The CLI only looks for it in ~/.kaggle, and it refuses to
# use the file unless the permissions are restricted to the owner.
from google.colab import files
uploaded = files.upload()

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

!pip install -q kaggle
!kaggle datasets download -d phuzoman/egg-producing-chickens
!unzip -q -o egg-producing-chickens.zip -d egg_data

In [ ]:
import pandas as pd
import glob

# The download unzips to a single CSV. glob finds it by pattern so the exact
# filename inside the zip does not have to be hardcoded.
csv_path = glob.glob('egg_data/*.csv')[0]
df = pd.read_csv(csv_path)
df.head()

In [ ]:
# Listing the column names here so the exact spelling and capitalisation used
# in the cells below can be checked against the real file.
print(df.columns.tolist())

## a. How many rows and columns are contained in the dataset? (3 marks)

In [ ]:
# shape returns the dimensions as a (rows, columns) tuple
num_rows, num_cols = df.shape

print("Rows:", num_rows)
print("Columns:", num_cols)

## b. What datatypes are contained in the dataset? (3 marks)

In [ ]:
# dtypes shows the type pandas inferred for each column when reading the CSV.
# object means text, int64/float64 are numeric.
df.dtypes

## c. Is the dataset complete? i.e. no missing values? (2 marks)

In [ ]:
# isnull() turns every cell into True/False for "is this empty", so summing
# down each column gives the number of missing values per column.
print(df.isnull().sum())

# Summing again collapses that to a single total across the whole dataset
print()
print("Total missing values:", df.isnull().sum().sum())

## d. Slice out the first 15 rows and last 20 rows, merge into `df_sample`, display it (4 marks)

In [ ]:
# head() takes rows from the top, tail() takes them from the bottom
first_15 = df.head(15)
last_20 = df.tail(20)

# concat stacks the two slices into one dataframe. Without ignore_index the
# original row numbers would carry over and the index would jump mid-way.
df_sample = pd.concat([first_15, last_20], ignore_index=True)

df_sample

## Notes / next steps
- `EggsPerDay` is the target for a regression model (predict expected daily yield). `AmountOfFeed`, `Age`, `GallusWeight`, and `SunLightExposure` are the strongest likely predictors based on the domain description.
- Categorical fields (breed, comb type, colors, plumage) will need encoding (one-hot / ordinal) before modeling.
- Because this dataset is synthetic, treat any accuracy numbers from it as a proof-of-concept for the ERP's forecasting module, not a production-ready model — real feed/production logs from the Nairobi-area farms in the study should replace or augment it before deployment.

---
## Additional exploration (beyond the required questions)
Not part of the graded a–d questions, but strengthens the case for this dataset feeding the ML engine.

### Duplicate rows check

In [ ]:
# duplicated() flags any row that is an exact repeat of an earlier one.
# Repeated rows would bias the model towards whatever they contain.
print("Duplicate rows:", df.duplicated().sum())

### Summary statistics for numeric columns

In [ ]:
# describe() summarises the numeric columns - count, mean, standard deviation,
# min, max and quartiles. Useful for spotting impossible values such as a
# negative weight or an egg count far outside the sensible range.
df.describe()

### Relationship between feed intake, age, and egg output

In [ ]:
import matplotlib.pyplot as plt

# Correlation against EggsPerDay shows which variables actually move with the
# target, which is what makes them worth keeping as predictors later.
numeric_cols = ['Age', 'GallusWeight', 'AmountOfFeed', 'SunLightExposure', 'EggsPerDay']
print(df[numeric_cols].corr()['EggsPerDay'].sort_values(ascending=False))

# A correlation value alone hides the shape of the relationship, so the two
# strongest predictors are also plotted to check whether it looks linear.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(df['AmountOfFeed'], df['EggsPerDay'], alpha=0.4)
axes[0].set_xlabel('Amount of Feed')
axes[0].set_ylabel('Eggs Per Day')
axes[0].set_title('Feed vs Eggs Per Day')

axes[1].scatter(df['Age'], df['EggsPerDay'], alpha=0.4)
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Eggs Per Day')
axes[1].set_title('Age vs Eggs Per Day')

plt.tight_layout()
plt.show()

### Breed distribution

In [ ]:
# Counting birds per breed. If one breed dominates the dataset, the model will
# fit that breed well and generalise poorly to the others.
df['GallusBreed'].value_counts()